# Portfolio Value at Risk (VaR) Engine

Computes portfolio VaR using three methods -- **Parametric (Variance-Covariance)**,
**Historical**, and **Monte Carlo** -- finds the portfolio weights that minimize
risk / minimize tail loss via constrained optimization, and runs a rolling
1-year backtest to see how VaR changes over time.

All calculation logic lives in `src/` (imported below); this notebook is the
narrative + display layer that calls into it, so the logic itself can be
tested and version-controlled independently of the write-up.

> **Known limitation (read before trusting the rolling-backtest numbers):**
> `src/backtest.py` has a documented look-ahead-bias issue in how rolling
> windows align with the "actual return" they're scored against -- see the
> module docstring and the note in Section 11 below. It has **not** been
> fixed in this version; the Kupiec test results in Section 12 should be
> read with that caveat in mind.


## 1. Setup

Import the calculation modules from `src/`, plus the libraries needed for
data loading and plotting at the notebook level.


In [ ]:
import sys
sys.path.append("..")  # so `src` is importable when running from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.data import load_close_prices, compute_returns
from src.portfolio_stats import scale_mu, scale_cov
from src.var_models import performance_portfolio, historical_var, monte_carlo_var
from src.optimize import find_min_risk_portfolio, find_min_loss_portfolio
from src.backtest import build_rolling_windows, compute_actual_return, calculate_var_metrics, kupiec_test
from src.report import show_portfolio, show_var_comparison


## 2. Config

All constants are defined in `src/config.py` rather than scattered as
"magic numbers" throughout the notebook, so a single parameter change
propagates everywhere it's used.

**Scaling convention** (used consistently throughout): volatility scales
with `sqrt(horizon)`, variance/covariance scales with `horizon` --
consistent with the i.i.d. daily returns assumption.


In [ ]:
print("Tickers          :", config.TICKERS)
print("Date range       :", config.START_DATE, "->", config.END_DATE)
print("Initial weights  :", config.W_PORT_INITIAL)
print("Horizon (days)   :", config.INVESTMENT_HORIZON)
print("Confidence       :", config.CONFIDENCE)
print("Rolling window   :", config.ROLLING_WINDOW, "days,", config.N_ROLLING_WINDOWS, "windows")


## 3. Load Price Data and Compute Returns

**`load_close_prices`**: fetches each ticker's daily close price from Yahoo
Finance and merges them into a single DataFrame. Each ticker is wrapped in
its own try/except -- if a given ticker cannot be fetched (e.g. delisted, or
rate-limited), a warning is printed and it is skipped instead of crashing
the whole run.

**`compute_returns`**: computes
- **Daily log return**: `ln(P_t / P_{t-1})` -- log returns are used because
  they are additive across time.
- **Annual simple return**: uses year-end close prices, simple return
  `(P_t/P_{t-1}) - 1`, to show a yearly overview of performance.


In [ ]:
close_prices = load_close_prices(config.TICKERS, config.START_DATE, config.END_DATE)
returns_daily, returns_annual = compute_returns(close_prices)
returns_daily.head()


## 4. Portfolio Statistics (Consistent Scaling)

`scale_mu` and `scale_cov` (from `src/portfolio_stats.py`) are the single
place where mean and covariance are scaled from daily to the desired
horizon -- used consistently everywhere in the project instead of writing
`* investment_horizon` in multiple scattered places.

- Mean scales linearly with horizon (`mu * horizon`)
- Covariance scales linearly with horizon as well (`cov * horizon`),
  because Var(n-day return) = n x Var(1-day return) under the i.i.d.
  returns assumption.


In [ ]:
mu_port = scale_mu(returns_daily.mean(), config.INVESTMENT_HORIZON)
sigma_port = scale_cov(returns_daily.cov(), config.INVESTMENT_HORIZON)
corr_port = returns_daily.corr()


## 5. Parametric VaR (Variance-Covariance Method)

`performance_portfolio` computes:
- **Portfolio return**: `w @ mu`
- **Portfolio risk (std)**: `sqrt(w @ sigma @ w)`
- **VaR**: `return + z * risk`, where `z = norm.ppf(1 - confidence)` is the
  left-tail quantile of the standard normal. E.g. confidence = 0.95 ->
  `z = norm.ppf(0.05) ~ -1.645`


## 6. Portfolio Optimization

Two functions (`src/optimize.py`) find portfolio weights via
`scipy.optimize.minimize` (SLSQP), subject to: weights sum to 1, and each
asset's weight lies within `[min_weight, max_weight]`.

**`find_min_risk_portfolio`** -- Global Minimum-Variance Portfolio
- Objective: minimize `sqrt(w @ sigma @ w)` only (expected return is
  ignored entirely).

**`find_min_loss_portfolio`** *(previously named `find_min_var_portfolio` --
renamed to reduce confusion)*
- VaR in this project is stored as a **negative** value (more negative =
  worse loss). So "maximizing VaR" mathematically is the same as
  "minimizing the magnitude of the tail loss". The old name made this easy
  to misread as "minimizing VaR" (which would mean approaching -infinity,
  the worst outcome) -- hence the rename.
- Objective: minimize `-(return + z * risk)` <=> maximize
  `(return + z * risk)` <=> minimize tail loss


## 7. Reporting Helpers

`show_portfolio` and `show_var_comparison` (`src/report.py`) are display
helpers, kept separate from calculation logic (separation of concerns) so
the logic can be tested/modified without affecting display, and vice
versa.


## 8. Run Parametric VaR

Compares three portfolios:
1. **initial** -- the manually specified starting weights (`W_PORT_INITIAL`)
2. **minrisk** -- the weights that minimize risk (portfolio std), ignoring
   return
3. **minloss** -- the weights that minimize tail loss (VaR), accounting
   for both return and risk via the VaR formula


In [ ]:
output_initial = performance_portfolio(config.W_PORT_INITIAL, mu_port, sigma_port, config.CONFIDENCE)
output_minrisk = find_min_risk_portfolio(mu_port, sigma_port, config.W_PORT_INITIAL, config.MIN_WEIGHT, config.MAX_WEIGHT, config.CONFIDENCE)
output_minloss = find_min_loss_portfolio(mu_port, sigma_port, config.W_PORT_INITIAL, config.MIN_WEIGHT, config.MAX_WEIGHT, config.CONFIDENCE)

show_portfolio("output_initial", output_initial, config.TICKERS)
show_portfolio("output_minrisk", output_minrisk, config.TICKERS)
show_portfolio("output_minloss", output_minloss, config.TICKERS)


## 9. Historical VaR

Finds VaR from the **empirical quantile** of actual historical returns,
without assuming returns are normally distributed.


In [ ]:
show_var_comparison(
    "VaR Historical (%)",
    {
        "initial": historical_var(returns_daily, output_initial["portfolio_weight"], 1 - config.CONFIDENCE),
        "minrisk": historical_var(returns_daily, output_minrisk["portfolio_weight"], 1 - config.CONFIDENCE),
        "minloss": historical_var(returns_daily, output_minloss["portfolio_weight"], 1 - config.CONFIDENCE),
    },
)


## 10. Monte Carlo VaR

Simulates correlated log-returns using **Cholesky decomposition** of the
correlation matrix, then finds the quantile of the **simulated portfolio**.

Steps:
1. `L = cholesky(corr)` -- decompose the correlation matrix to generate
   correlated shocks
2. Draw `Z_independent` as independent standard normals, then transform to
   `Z_correlated = Z_independent @ L.T`
3. Simulate each asset's log return via the GBM formula:
   `(mu - 0.5*sigma^2) + sigma*Z_correlated`
4. Combine into the portfolio return: `log_returns @ w_port`
5. Take the quantile of the **simulated portfolio** (not the quantile of
   each individual asset)


In [ ]:
mu_daily = returns_daily.mean().to_numpy()
sigma_daily = returns_daily.std().to_numpy()
corr_matrix = corr_port.to_numpy()

mc_kwargs = dict(
    mu_daily=mu_daily,
    sigma_daily=sigma_daily,
    corr=corr_matrix,
    horizon=config.INVESTMENT_HORIZON,
    alpha=1 - config.CONFIDENCE,
    n_sims=config.MC_SIMS_STATIC,
)

show_var_comparison(
    "VaR Monte Carlo (%)",
    {
        "initial": monte_carlo_var(w_port=output_initial["portfolio_weight"], **mc_kwargs),
        "minrisk": monte_carlo_var(w_port=output_minrisk["portfolio_weight"], **mc_kwargs),
        "minloss": monte_carlo_var(w_port=output_minloss["portfolio_weight"], **mc_kwargs),
    },
)


## 11. Rolling 1-Year Re-estimation

Recomputes all three VaR methods on a **rolling 252-day window** (~1
trading year), stepped by 1 day at a time, to see how VaR changes over
time (e.g. VaR should worsen during periods of high market volatility).

**Index convention**: key `0` = the most recent window (today, looking
back 252 days), key `-i` = the window shifted back `i` days from today.

> **Caveat -- not fixed in this version:** `build_rolling_windows` and
> `compute_actual_return` currently overlap by one day -- the window used
> to *estimate* VaR for key `-i` includes the same day used to *score*
> that VaR. This is look-ahead bias and likely makes the models look more
> accurate than they are. See the docstring in `src/backtest.py` for the
> exact mechanism and the untaken fix. Left as-is at the user's request;
> revisit before drawing conclusions from Section 12.


In [ ]:
returns_daily_1Y = build_rolling_windows(returns_daily, config.ROLLING_WINDOW, config.N_ROLLING_WINDOWS)

actual_return = compute_actual_return(returns_daily, output_initial["portfolio_weight"], config.N_ROLLING_WINDOWS)
output_rolling = calculate_var_metrics(
    returns_daily_1Y,
    output_initial["portfolio_weight"],
    config.N_ROLLING_WINDOWS,
    config.CONFIDENCE,
    t=1,
    mc_sims=config.MC_SIMS_ROLLING,
)

result_var = pd.DataFrame({
    "Actual Return": actual_return["actual_return"],
    "Parametric VaR": output_rolling["Parametric VaR"],
    "Historical VaR": output_rolling["Historical VaR"],
    "Monte VaR": output_rolling["Monte VaR"],
})

result_var


In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(result_var.index, result_var["Actual Return"], label="Actual Return", linestyle="-", linewidth=2.5)
plt.plot(result_var.index, result_var["Parametric VaR"], label="Parametric VaR", linestyle="--", linewidth=2.5)
plt.plot(result_var.index, result_var["Historical VaR"], label="Historical VaR", linestyle="--", linewidth=2.5)
plt.plot(result_var.index, result_var["Monte VaR"], label="Monte VaR", linestyle="--", linewidth=2.5)

plt.title("Model VaR", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("return", fontsize=12)
plt.margins(x=0)

plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


## 12. Validating VaR Models with Kupiec's POF Test (Backtesting)

`kupiec_test` (`src/backtest.py`) performs **backtesting** to evaluate how
accurately each of the three VaR models (Parametric, Historical, and Monte
Carlo) predicts risk, using the **Kupiec Proportion of Failures (POF)
test**, based on the binomial distribution.

---

### 1. Statistical formulation

The test asks whether the **number of times the actual loss was worse
than the model's predicted VaR (exceptions, $x$)** is consistent, in a
statistically significant sense, with the stated confidence level ($c$).

*   **Null hypothesis ($H_0$):** the model is accurate -- the observed
    exception rate ($\hat{p}$) equals the theoretical rate ($p = 1 - c$)
*   **Alternative hypothesis ($H_1$):** the model is inaccurate (it may
    underestimate or overestimate risk)

The Likelihood Ratio test statistic ($LR_{pof}$) is:
$$LR_{pof} = -2 \ln \left[ \frac{(1-p)^{N-x} p^x}{(1-\hat{p})^{N-x} \hat{p}^x} \right]$$

---

### 2. Line-by-line breakdown

*   **`x = sum(Actual Return < Model VaR)`** -- counts how many times
    `Actual Return` was less than `Model VaR` (i.e. the actual loss was
    worse than predicted, since VaR is stored as a negative value in this
    project)
*   **`p = 1 - confidence`** -- the theoretical exception rate
    (significance level, $\alpha$). E.g. at 95% confidence, $p = 0.05$
*   **`p_hat = x / N`** -- the observed exception rate from the data
*   **`lr_pof = -2 * log(...)`** -- the Likelihood Ratio statistic,
    compared against a Chi-Square distribution with 1 degree of freedom
*   **`Crit = chi2.ppf(1 - p, df=1)`** -- the critical value at
    significance level $\alpha = p = 1-\text{confidence}$
*   **`p_value = 1 - chi2.cdf(lr_pof, df=1)`** -- the p-value, used to
    decide between the hypotheses

---

### 3. Decision rule

Compare **the p-value against the significance level $\alpha$** (not
against confidence):

| Statistical result | Decision | What it means for the model |
| :--- | :--- | :--- |
| **p-value $\ge \alpha$** | **Fail to reject $H_0$** | **Model passes** |
| **p-value $< \alpha$** | **Reject $H_0$** | **Model fails** |

---

### 4. Important caveat for this run

With `N_ROLLING_WINDOWS = 252`, the expected number of exceptions at 95%
confidence is only ~12.6. The Kupiec test has **low statistical power** at
this sample size -- it may fail to distinguish "the model is genuinely
inaccurate" from "this is normal sampling variation." Treat a "Pass" here
as weak evidence, not proof of model adequacy. Combine with the
look-ahead-bias caveat from Section 11 before drawing conclusions.


In [ ]:
print("--- Parametric VaR ---")
kupiec_test(output_rolling["Parametric VaR"], actual_return["actual_return"], config.CONFIDENCE)

print("\n--- Historical VaR ---")
kupiec_test(output_rolling["Historical VaR"], actual_return["actual_return"], config.CONFIDENCE)

print("\n--- Monte Carlo VaR ---")
kupiec_test(output_rolling["Monte VaR"], actual_return["actual_return"], config.CONFIDENCE)
